In [1]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

In [2]:
# Caminho para o shapefile
shp_path = r"..\..\Data\Processed\PT-FireSprd_v3.0\L2_FireBehavior\PT-FireSprd_v3.0_L2_model.shp"

# 1. Ler o shapefile
gdf = gpd.read_file(shp_path)

# 4. Listar as variáveis restantes
print("\n=== Variáveis mantidas para análise ===")
print(gdf.columns.tolist())


=== Variáveis mantidas para análise ===
['ros_p', 'duration_p', 'elev_av', 'aspect_sin', 'aspect_cos', 'landform', 'land_use', '1_3y_fir_p', '3_8y_fir_p', '8_ny_fir_p', 'fuel_age', 'fuel_model', 'f_load_av', 'sW_1m_av', 'sW_3m_av', 'sW_7_av', 'sW_28_av', 'sW_100_av', 'sW_289_av', 't_2m_C_av', 'd_2m_C_av', 'rh_2m_av', 'VPD_Pa_av', 'sP_hPa_av', 'gp_m2s2_av', 'dfmc_av', 'HDW_av', 'Haines_av', 'FWI_12h_av', 'DC_12h_av', 'FFMC_12h_a', 'ISI_12h_av', 'wv10_kh_av', 'wsin10_av', 'wcos10_av', 'wv100_k_av', 'wsin100_av', 'wcos100_av', 'Recirc', 'CircVar', 't_950_av', 't_850_av', 't_700_av', 't_500_av', 't_300_av', 'rh_950_av', 'rh_850_av', 'rh_700_av', 'rh_500_av', 'rh_300_av', 'wv_950_av', 'wv_850_av', 'wv_700_av', 'wv_500_av', 'wv_300_av', 'wsi_950_av', 'wco_950_av', 'wsi_850_av', 'wco_850_av', 'wsi_700_av', 'wco_700_av', 'wsi_500_av', 'wco_500_av', 'wsi_300_av', 'wco_300_av', 'vwv_950_av', 'vwv_850_av', 'vwv_700_av', 'vwv_500_av', 'vwv_300_av', 'gp_950_av', 'gp_850_av', 'gp_700_av', 'gp_500_a

In [3]:
# 3. Selecionar apenas as variáveis numéricas
num_df = gdf.select_dtypes(include=["number"])

# 4. Calcular a matriz de correlação
corr_matrix = num_df.corr()


In [4]:
plt.figure(
    figsize=(
        max(10, len(corr_matrix.columns)),   # largura proporcional ao nº de variáveis
        max(10, 0.95 * len(corr_matrix.columns))    # altura menor para ficar mais "quadrado"
    )
)

ax = sns.heatmap(
    corr_matrix,
    cmap='coolwarm',
    center=0,
    annot=True,                 # ativa os valores nas células
    fmt=".2f",                  # formato (2 casas decimais)
    annot_kws={"size": 10},      # tamanho da fonte dos labels
    linewidths=0.3,
    cbar_kws={"shrink": 0.8}
)


plt.title(
    "Matriz de Correlação das Variáveis Ambientais e ROS",
    fontsize=136,   # título maior
    pad=140
)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(
    labelsize=72,    # tamanho da fonte
    length=30,       # comprimento dos ticks (aumenta o espaço visual)
    width=5,        # largura dos ticks
    direction='out'  # ou 'in' para ticks para dentro
)
# Rótulos maiores e mais legíveis
plt.xticks(rotation=90, fontsize=39)
plt.yticks(rotation=0, fontsize=39)

plt.tight_layout()
plt.savefig("..\..\Data\Data_Exploration\Correlation_Matrix.pdf", format="pdf", dpi=300, bbox_inches="tight")
plt.show()


In [5]:
import numpy as np
import pandas as pd

n = 200
pd.set_option("display.max_rows", n)  


# Criar matriz sem diagonal
corr_mod = corr_matrix.abs().where(~np.eye(len(corr_matrix), dtype=bool))

# Manter só metade superior da matriz (remove duplicados)
corr_upper = corr_mod.where(np.triu(np.ones(corr_mod.shape), k=1).astype(bool))

# Transformar em DataFrame limpo
df_corr = (
    corr_upper
    .stack()
    .sort_values(ascending=False)
    .reset_index()
)

df_corr.columns = ["Var1", "Var2", "Correlation"]

# Pegar somente top 50
top50 = df_corr.head(n)

print(top50)


           Var1        Var2  Correlation
0        Cin_av     EL_m_av     1.000000
1     Haines_av   gT_8_7_av     1.000000
2     gT_s_9_av      Cin_av     0.994533
3      sW_3m_av   sW_289_av     0.993971
4      sW_1m_av   sW_100_av     0.993138
5      rh_2m_av     dfmc_av     0.990399
6     sP_hPa_av  gp_m2s2_av     0.987548
7    wcos100_av  wco_950_av     0.976537
8    wsin100_av  wsi_950_av     0.976243
9     wcos10_av  wcos100_av     0.975291
10    wsin10_av  wsin100_av     0.974195
11    wcos10_av  wco_950_av     0.965130
12    wsin10_av  wsi_950_av     0.955702
13    gp_700_av   gp_500_av     0.950640
14     rh_2m_av  LCL_hPa_av     0.949564
15     sW_3m_av   sW_100_av     0.946462
16   LFC_hPa_av  CCL_hPa_av     0.944678
17    t_2m_C_av   VPD_Pa_av     0.943644
18     sW_1m_av    sW_28_av     0.942233
19   HigCC_p_av  TotCC_p_av     0.937789
20    gp_500_av   gp_300_av     0.937046
21       Recirc     CircVar     0.933298
22      dfmc_av  LCL_hPa_av     0.932983
23     sW_1m_av 